In [9]:
import sys
sys.path.append('..')

from agents.graph import xai_agent
print("agent imported")

agent imported


In [10]:
import torch
from torchvision import transforms
from PIL import Image
import glob

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

image_files = glob.glob("../data/HAM10000_images_part_1/*.jpg")
test_path   = image_files[0]
test_tensor = transform(Image.open(test_path).convert("RGB")).unsqueeze(0)

print("test image:", test_path)
print("tensor shape:", test_tensor.shape)

test image: ../data/HAM10000_images_part_1\ISIC_0024306.jpg
tensor shape: torch.Size([1, 3, 64, 64])


In [11]:
initial_state = {
    "image_path":      test_path,
    "image_tensor":    test_tensor,
    "prediction":      "nv",
    "confidence":      0.99,
    "pred_class_idx":  0,
    "shap_result":     {},
    "gradcam_result":  {},
    "critique":        "",
    "contradictions":  [],
    "next_action":     "",
    "explanation":     "",
    "confidence_note": "",
    "loop_count":      0
}

print("running agent on high confidence image (0.99)...")
print("expected: planner skips SHAP, goes straight to gradcam\n")

result = xai_agent.invoke(initial_state)

running agent on high confidence image (0.99)...
expected: planner skips SHAP, goes straight to gradcam

  [Planner] confidence=0.99, loop=0
  [Planner router] high confidence, going gradcam only
  [GradCAM node] running Grad-CAM analysis...
  [GradCAM node] done. status=success, region=top-center region of the image
  [Critic node] reviewing SHAP and Grad-CAM results...
  [Critic node] contradictions=1, loop_count=0
    - SHAP analysis failed or returned error
  [Critic router] contradiction found, looping back to planner
  [Planner] confidence=0.99, loop=1
  [Planner router] high confidence, going gradcam only
  [GradCAM node] running Grad-CAM analysis...
  [GradCAM node] done. status=success, region=top-center region of the image
  [Critic node] reviewing SHAP and Grad-CAM results...
  [Critic node] contradictions=1, loop_count=1
    - SHAP analysis failed or returned error
  [Critic router] no contradiction or max loops, going to narrator
  [Narrator node] writing explanation...
  

In [12]:
print("prediction:      ", result['prediction'])
print("confidence:      ", result['confidence'])
print("next_action:     ", result['next_action'])
print("critique:        ", result['critique'])
print("contradictions:  ", result['contradictions'])
print("confidence_note: ", result['confidence_note'])
print("\nexplanation:")
print(result['explanation'])

prediction:       nv
confidence:       0.99
next_action:      gradcam_only
critique:         Issues found: SHAP analysis failed or returned error
contradictions:   ['SHAP analysis failed or returned error']
confidence_note:  Low confidence - contradictions found, human review recommended

explanation:
The model predicted 'nv' with 99.0% confidence. SHAP analysis shows: not available. Grad-CAM shows the model focused on the top-center region of the image. The model's attention was broadly distributed across much of the image with high gradient signal strength. High-attention coverage: 48.7% of image area. Critic review: Issues found: SHAP analysis failed or returned error.


In [14]:
import importlib
import agents.shap_node as sn
importlib.reload(sn)
import agents.graph as gm
importlib.reload(gm)
from agents.graph import xai_agent
print("reloaded")

XAI agent graph compiled successfully
reloaded


In [15]:
low_conf_state = {
    "image_path":      test_path,
    "image_tensor":    test_tensor,
    "prediction":      "mel",
    "confidence":      0.45,
    "pred_class_idx":  1,
    "shap_result":     {},
    "gradcam_result":  {},
    "critique":        "",
    "contradictions":  [],
    "next_action":     "",
    "explanation":     "",
    "confidence_note": "",
    "loop_count":      0
}

print("running agent on low confidence image (0.45)...")
print("expected: planner runs SHAP first, then gradcam\n")

result2 = xai_agent.invoke(low_conf_state)

print("\nexplanation:")
print(result2['explanation'])
print("\nconfidence note:", result2['confidence_note'])

running agent on low confidence image (0.45)...
expected: planner runs SHAP first, then gradcam

  [Planner] confidence=0.45, loop=0
  [Planner router] low confidence, going shap first
  [SHAP node] running SHAP analysis...
  [SHAP node] background loaded: torch.Size([10, 3, 64, 64])
  [SHAP node] done. status=error, mean=0.000000
  [GradCAM node] running Grad-CAM analysis...
  [GradCAM node] done. status=success, region=top-center region of the image
  [Critic node] reviewing SHAP and Grad-CAM results...
  [Critic node] contradictions=1, loop_count=0
    - SHAP analysis failed or returned error
  [Critic router] contradiction found, looping back to planner
  [Planner] confidence=0.45, loop=1
  [Planner router] low confidence, going shap first
  [SHAP node] running SHAP analysis...
  [SHAP node] done. status=error, mean=0.000000
  [GradCAM node] running Grad-CAM analysis...
  [GradCAM node] done. status=success, region=top-center region of the image
  [Critic node] reviewing SHAP and G

In [16]:
print("high confidence test:")
print("  shap ran:", result['shap_result'].get('status', 'skipped'))
print("  gradcam ran:", result['gradcam_result'].get('status', 'not run'))

print("\nlow confidence test:")
print("  shap ran:", result2['shap_result'].get('status', 'skipped'))
print("  gradcam ran:", result2['gradcam_result'].get('status', 'not run'))

print("\nrouting is working correctly if:")
print("  high confidence -> shap=skipped, gradcam=success")
print("  low confidence  -> shap=success,  gradcam=success")

high confidence test:
  shap ran: skipped
  gradcam ran: success

low confidence test:
  shap ran: error
  gradcam ran: success

routing is working correctly if:
  high confidence -> shap=skipped, gradcam=success
  low confidence  -> shap=success,  gradcam=success
